In [ ]:
# !pip install transformers torch

In [1]:
import os
os.environ['TRANSFORMERS_CACHE'] = '/home/david.yang1/.cache/huggingface/'
os.environ['HF_HOME'] = '/home/david.yang1/.cache/huggingface/'

In [2]:
from transformers import AutoTokenizer, AutoModel, pipeline, AutoModelForSequenceClassification, Trainer, TrainingArguments
import torch
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from collections import defaultdict
import pandas as pd
from datasets import Dataset, DatasetDict
from huggingface_hub import login
import evaluate
from ray.tune.search.hyperopt import HyperOptSearch
from ray.tune.schedulers import ASHAScheduler
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

/home/david.yang1/.local/lib/python3.11/site-packages/transformers/utils/hub.py:124: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
2024-06-12 14:58:58.300367: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2024-06-12 14:58:58.344466: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-06-12 14:58:58.344506: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-06-12 14:58:58.344544: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
20

In [3]:
login()

# Load and inspect data

In [4]:
df = pd.read_csv('bn_pub_dataset_3.csv')

In [5]:
# Check class distribution
print(df['label'].value_counts())

# Balance classes if needed
df = df.groupby('label').sample(n=min(df['label'].value_counts()), random_state=42)

# Shuffle the dataset
df = df.sample(frac=1, random_state=42)
df = df[["text", "label"]]

# Split dataset into test & train
df_train = df[128:]
df_val = df[:64]
df_test = df[64:128]

label
0    309
1    309
Name: count, dtype: int64


In [6]:
# ds = Dataset.from_pandas(df)
# train_test_valid = ds.train_test_split(test_size=0.35)
# test_valid = train_test_valid['test'].train_test_split(test_size=0.5)
# train_test_valid_dataset = DatasetDict({
#     'train': train_test_valid['train'],
#     'test': test_valid['test'],
#     'valid': test_valid['train']})

# train_test_valid_dataset

In [7]:
tds = Dataset.from_pandas(df_train)
vds = Dataset.from_pandas(df_val)
test = Dataset.from_pandas(df_test)

# ds = DatasetDict()
# ds["train"] = tds
# ds["validation"] = vds
# ds

https://medium.com/@fhirfly/fine-tuning-biobert-v1-1-on-a-large-dataset-classifying-medical-queries-c33b4d08ec6a

# Preprocess text dataset

In [8]:
# Load BioBERT model and tokenizer
tokenizer = AutoTokenizer.from_pretrained('dmis-lab/biobert-v1.1')
# model = AutoModel.from_pretrained('dmis-lab/biobert-v1.1')

/home/david.yang1/.local/lib/python3.11/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [9]:
# Tokenize the data
def tokenize_function(df):
    return tokenizer(
        df['text'],
        padding="longest",
        truncation=True,
        max_length = 512
    )

# Apply the tokenizer to the datasets
tds = tds.map(tokenize_function, batched=True)
vds = vds.map(tokenize_function, batched=True)
test = test.map(tokenize_function, batched=True)

Map:   0%|          | 0/490 [00:00<?, ? examples/s]

Map:   0%|          | 0/64 [00:00<?, ? examples/s]

Map:   0%|          | 0/64 [00:00<?, ? examples/s]

In [10]:
# Set the format of the datasets to include only the required columns
tds = tds.rename_column('__index_level_0__', 'index').remove_columns(['text', 'index'])
vds = vds.rename_column('__index_level_0__', 'index').remove_columns(['text', 'index'])
test = test.rename_column('__index_level_0__', 'index').remove_columns(['text', 'index'])

# Define DatasetDict
ds = DatasetDict({
    "train": tds,
    "validation": vds,
    "test": test
})

# Model fine tuning
Parameter tuning: 
https://kaitchup.substack.com/p/a-guide-on-hyperparameters-and-training
https://medium.com/distributed-computing-with-ray/hyperparameter-optimization-for-transformers-a-guide-c4e32c6c989b
https://huggingface.co/blog/ray-tune


In [11]:
# Load the pre-trained model
# bn_model = AutoModelForSequenceClassification.from_pretrained('dmis-lab/biobert-v1.1', num_labels=2)

def model_init():
    return AutoModelForSequenceClassification.from_pretrained('dmis-lab/biobert-v1.1', num_labels=2)

In [12]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)

    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

In [13]:
# Define the training arguments
training_args = TrainingArguments(
    output_dir='./results',
    evaluation_strategy = "steps",
    eval_steps=500,
    num_train_epochs=3,    # number of training epochs
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    warmup_ratio=0.01,
    weight_decay=0.01,
    logging_dir='./logs',
)
# Create the Trainer and start training
# trainer = Trainer(
#     model=bn_model,
#     args=training_args,
#     train_dataset=ds["train"],
#     eval_dataset=ds["validation"]
# )

In [14]:
trainer = Trainer(
    args=training_args,
    train_dataset=ds["train"],
    eval_dataset=ds["validation"],
    model_init=model_init,
    compute_metrics=compute_metrics,
)

/home/david.yang1/.local/lib/python3.11/site-packages/accelerate/accelerator.py:446: FutureWarning: Passing the following arguments to `Accelerator` is deprecated and will be removed in version 1.0 of Accelerate: dict_keys(['dispatch_batches', 'split_batches', 'even_batches', 'use_seedable_sampler']). Please pass an `accelerate.DataLoaderConfiguration` instead: 
dataloader_config = DataLoaderConfiguration(dispatch_batches=None, split_batches=False, even_batches=True, use_seedable_sampler=True)
  warnings.warn(
Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at dmis-lab/biobert-v1.1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inferen

In [15]:
trainer.train()

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at dmis-lab/biobert-v1.1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step,Training Loss,Validation Loss


TrainOutput(global_step=93, training_loss=0.3241984459661668, metrics={'train_runtime': 846.3203, 'train_samples_per_second': 1.737, 'train_steps_per_second': 0.11, 'total_flos': 386773251379200.0, 'train_loss': 0.3241984459661668, 'epoch': 3.0})

In [16]:
trainer.state.log_history

[{'train_runtime': 846.3203,
  'train_samples_per_second': 1.737,
  'train_steps_per_second': 0.11,
  'total_flos': 386773251379200.0,
  'train_loss': 0.3241984459661668,
  'epoch': 3.0,
  'step': 93}]

In [17]:
# validation_results = pd.DataFrame(columns=["accuracy", "f1", "precision", "recall"])
# validation_results

In [18]:
# Evaluate the model
eval = trainer.evaluate(ds["test"])

In [19]:
eval_2 = trainer.evaluate(ds["validation"])

In [20]:
eval

{'eval_loss': 0.5576477646827698,
 'eval_accuracy': 0.828125,
 'eval_f1': 0.8405797101449276,
 'eval_precision': 0.8055555555555556,
 'eval_recall': 0.8787878787878788,
 'eval_runtime': 8.8061,
 'eval_samples_per_second': 7.268,
 'eval_steps_per_second': 0.114,
 'epoch': 3.0}

In [21]:
eval_2

{'eval_loss': 0.4342470169067383,
 'eval_accuracy': 0.84375,
 'eval_f1': 0.8484848484848486,
 'eval_precision': 0.8,
 'eval_recall': 0.9032258064516129,
 'eval_runtime': 8.846,
 'eval_samples_per_second': 7.235,
 'eval_steps_per_second': 0.113,
 'epoch': 3.0}

In [ ]:
pred = trainer.predict(ds["test"])

In [ ]:
pred

In [ ]:
print(pred.metrics["test_confusion_matrix"])

In [ ]:
y_pred = pred.predictions

In [ ]:
predictions = y_pred.argmax(axis=-1)

In [ ]:
metric.compute(references = pred.label_ids, predictions=predictions)

# Hyperparameter search

In [ ]:
# Default objective is the sum of all metrics
# when metrics are provided, so we have to maximize it.
trainer.hyperparameter_search(
    direction="maximize", 
    backend="ray", 
    n_trials=10 # number of trials
)

In [ ]:
best_trial = trainer.hyperparameter_search(
    direction="maximize",
    backend="ray",
    search_alg=HyperOptSearch(metric="objective", mode="max"),
    scheduler=ASHAScheduler(metric="objective", mode="max")
)

In [ ]:
# Evaluate the model
trainer.evaluate(ds["validation"])

# ARCHIVE

In [ ]:
# def encode_data(tokenizer, text, max_length):
#     encoded = tokenizer.batch_encode_plus(
#         text,
#         truncation=True,
#         padding='longest',
#         max_length=max_length,
#         return_tensors='pt'  # return PyTorch tensors
#     )
#     return encoded["input_ids"], encoded["attention_mask"]
# # Use an appropriate max_length 
# input_ids_train, attention_mask_train = encode_data(tokenizer, df_train['text'].tolist(), max_length=512)
# input_ids_val, attention_mask_val = encode_data(tokenizer, df_val['text'].tolist(), max_length=512)